In [ ]:
from pathlib import Path
from datetime import datetime
import shutil

import polars as pl
from pokerkit import HandHistory

# -------------------------
# Paths
# -------------------------
ROOT = Path("handhq")
OUTPUT = Path("data")

HANDS_DIR = OUTPUT / "hands"
PLAYER_HANDS_DIR = OUTPUT / "player_hands"
ACTIONS_DIR = OUTPUT / "actions"

# -------------------------
# Full-dataset settings
# -------------------------
MAX_FILES = None          # None = process every .phhs file
FILES_PER_CHUNK = 100     # keeps memory usage manageable
RESET_OUTPUT = True       # removes old generated parquet chunks before starting

if RESET_OUTPUT:
    for directory in (HANDS_DIR, PLAYER_HANDS_DIR, ACTIONS_DIR):
        if directory.exists():
            shutil.rmtree(directory)

for directory in (HANDS_DIR, PLAYER_HANDS_DIR, ACTIONS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Explicit schemas keep parquet chunk types consistent across the full dataset.
HANDS_SCHEMA = {
    "hand_id": pl.String,
    "variant": pl.String,
    "venue": pl.String,
    "table": pl.String,
    "datetime": pl.Datetime,
    "currency": pl.String,
    "min_bet": pl.Float64,
    "small_bet": pl.Float64,
    "big_bet": pl.Float64,
    "bring_in": pl.Float64,
    "players_dealt": pl.Int64,
    "seat_count": pl.Int64,
    "has_winnings": pl.Boolean,
    "has_finishing_stacks": pl.Boolean,
    "source_file": pl.String,
}

PLAYER_SCHEMA = {
    "hand_id": pl.String,
    "player_id": pl.String,
    "player_number": pl.Int64,
    "seat": pl.Int64,
    "starting_stack": pl.Float64,
    "ante": pl.Float64,
    "blind_or_straddle": pl.Float64,
    "winnings": pl.Float64,
}

ACTION_SCHEMA = {
    "hand_id": pl.String,
    "action_number": pl.Int64,
    "raw_action": pl.String,
}


def to_float(value):
    if value is None:
        return None
    return float(value)


def make_datetime(hand):
    if hand.year is None or hand.month is None or hand.day is None:
        return None

    hour = hand.time.hour if hand.time is not None else 0
    minute = hand.time.minute if hand.time is not None else 0
    second = hand.time.second if hand.time is not None else 0

    return datetime(
        hand.year,
        hand.month,
        hand.day,
        hour,
        minute,
        second,
    )


def get_player_value(values, i):
    if values is None or i >= len(values):
        return None
    return values[i]


def extract_hand_rows(hand, source_file):
    num_players = len(hand.starting_stacks)

    hand_row = {
        "hand_id": str(hand.hand),
        "variant": hand.variant,
        "venue": hand.venue,
        "table": hand.table,
        "datetime": make_datetime(hand),
        "currency": hand.currency,
        "min_bet": to_float(hand.min_bet),
        "small_bet": to_float(hand.small_bet),
        "big_bet": to_float(hand.big_bet),
        "bring_in": to_float(hand.bring_in),
        "players_dealt": num_players,
        "seat_count": hand.seat_count,
        "has_winnings": hand.winnings is not None,
        "has_finishing_stacks": hand.finishing_stacks is not None,
        "source_file": str(source_file),
    }

    player_rows = []
    for i in range(num_players):
        player_rows.append({
            "hand_id": str(hand.hand),
            "player_id": get_player_value(hand.players, i),
            "player_number": i + 1,
            "seat": get_player_value(hand.seats, i),
            "starting_stack": to_float(get_player_value(hand.starting_stacks, i)),
            "ante": to_float(get_player_value(hand.antes, i)),
            "blind_or_straddle": to_float(get_player_value(hand.blinds_or_straddles, i)),
            "winnings": to_float(get_player_value(hand.winnings, i)),
        })

    action_rows = [
        {
            "hand_id": str(hand.hand),
            "action_number": action_number,
            "raw_action": action,
        }
        for action_number, action in enumerate(hand.actions)
    ]

    return hand_row, player_rows, action_rows


def process_chunk(files, chunk_number):
    hands_rows = []
    player_rows = []
    action_rows = []
    total_hands = 0

    for file_path in files:
        try:
            with open(file_path, "rb") as f:
                for hand in HandHistory.load_all(f):
                    hand_row, players, actions = extract_hand_rows(hand, file_path)
                    hands_rows.append(hand_row)
                    player_rows.extend(players)
                    action_rows.extend(actions)
                    total_hands += 1
        except Exception as e:
            print(f"\nERROR reading {file_path}: {e}")

    if hands_rows:
        pl.DataFrame(hands_rows, schema=HANDS_SCHEMA, strict=False).write_parquet(
            HANDS_DIR / f"hands_{chunk_number:04d}.parquet",
            compression="zstd",
        )

    if player_rows:
        pl.DataFrame(player_rows, schema=PLAYER_SCHEMA, strict=False).write_parquet(
            PLAYER_HANDS_DIR / f"player_hands_{chunk_number:04d}.parquet",
            compression="zstd",
        )

    if action_rows:
        pl.DataFrame(action_rows, schema=ACTION_SCHEMA, strict=False).write_parquet(
            ACTIONS_DIR / f"actions_{chunk_number:04d}.parquet",
            compression="zstd",
        )

    return total_hands


# -------------------------
# Process all PHH files
# -------------------------
files = sorted(ROOT.rglob("*.phhs"))

if MAX_FILES is not None:
    files = files[:MAX_FILES]

if not files:
    raise FileNotFoundError(
        f"No .phhs files found under {ROOT.resolve()}"
    )

num_chunks = (len(files) + FILES_PER_CHUNK - 1) // FILES_PER_CHUNK

grand_total_hands = 0

for chunk_number, start in enumerate(range(0, len(files), FILES_PER_CHUNK)):
    chunk = files[start:start + FILES_PER_CHUNK]
    print(
        f"Processing chunk {chunk_number + 1}/{num_chunks}",
        end="\r",
        flush=True,
    )
    grand_total_hands += process_chunk(chunk, chunk_number)

print(" " * 60, end="\r")

# -------------------------
# Read the generated parquet dataset lazily
# -------------------------
hands = pl.scan_parquet(str(HANDS_DIR / "*.parquet"))
players = pl.scan_parquet(str(PLAYER_HANDS_DIR / "*.parquet"))
actions = pl.scan_parquet(str(ACTIONS_DIR / "*.parquet"))

# -------------------------
# Final output: heads + row totals only
# -------------------------
print("=== HANDS HEAD ===")
print(hands.head(5).collect())

print("\n=== PLAYER_HANDS HEAD ===")
print(players.head(5).collect())

print("\n=== ACTIONS HEAD ===")
print(actions.head(5).collect())

counts = pl.collect_all([
    hands.select(pl.len().alias("hands_rows")),
    players.select(pl.len().alias("player_hand_rows")),
    actions.select(pl.len().alias("action_rows")),
])

print("\n=== TOTAL ROWS ===")
print(f"hands:        {counts[0][0, 'hands_rows']:,}")
print(f"player_hands: {counts[1][0, 'player_hand_rows']:,}")
print(f"actions:      {counts[2][0, 'action_rows']:,}")
